# Chapter 2: Searching State Spaces

```{admonition} Learning Objectives
:class: tip
- Understand state space representation as graphs
- Master uninformed search algorithms: BFS, DFS, UCS, Iterative Deepening
- Master informed search algorithms: Greedy Best-First, A*
- Apply local search methods: Hill Climbing, Simulated Annealing, Genetic Algorithms
- Solve Constraint Satisfaction Problems with various techniques
- Analyze algorithm complexity, optimality, and completeness
- Implement and compare different search strategies on real problems
```

```{epigraph}
The formulation of a problem is often more essential than its solution.

-- Albert Einstein
```

## 2.1 Introduction

Search is one of the most fundamental problem-solving techniques in artificial intelligence. Many AI problems can be formulated as finding a sequence of actions that transforms an initial state into a goal state. This paradigm applies to diverse applications:

- **Route Planning**: Finding optimal paths in maps (GPS navigation)
- **Puzzle Solving**: 8-puzzle, Rubik's cube, Sudoku
- **Game Playing**: Chess, checkers, Go (searching game trees)
- **Scheduling**: Resource allocation, task planning
- **Robot Motion Planning**: Navigation in physical space
- **Theorem Proving**: Finding proofs in logic

### Why Search Matters

Search is crucial because:
1. Many problems have no known direct solution formula
2. The solution space is too large to enumerate
3. We need systematic methods to explore possibilities
4. Different search strategies have different trade-offs (time, memory, optimality)

### Search vs. Planning

- **Search**: Finding a path through a state space (focuses on *how* to reach goal)
- **Planning**: Constructing a sequence of actions (focuses on *what* actions to take)

In practice, the distinction blurs - both involve exploring state spaces to find solutions.

### 2.1.1 State Space as a Graph

The **state space** of a problem is modeled as a directed graph:

**Components:**

1. **States (Nodes)**: Represent configurations of the world
   - Example (8-puzzle): Arrangement of tiles
   - Example (route planning): Geographic locations

2. **Actions (Edges)**: Transitions between states
   - Example (8-puzzle): Slide tile up/down/left/right
   - Example (route planning): Drive from city A to city B

3. **Initial State**: Starting configuration
   - Where the agent begins

4. **Goal State(s)**: Desired configuration(s)
   - What we're trying to achieve
   - Can be explicit state or test function

5. **Path Cost**: Sum of action costs in a sequence
   - Often uniform (1 per action) or weighted (different costs)

**State Space Graph Properties:**

- **Branching Factor (b)**: Average number of successors per state
- **Depth (d)**: Distance from initial state to shallowest goal
- **Maximum Depth (m)**: Longest path in state space
- **Size**: Total number of reachable states

These properties determine algorithm performance.

### Search Tree vs. State Space Graph

**State Space Graph:**
- Each state appears once
- Captures actual problem structure
- May contain cycles

**Search Tree:**
- States can repeat on different paths
- Root is initial state
- Represents exploration process
- Can be infinite even if state space is finite (due to cycles)

Most search algorithms work on the search tree but avoid revisiting states using a visited/explored set.

## 2.2 Uninformed Search Algorithms

**Uninformed** (or **blind**) search strategies use only information available in the problem definition. They don't use problem-specific knowledge to guide the search.

Key uninformed strategies:
1. Breadth-First Search (BFS)
2. Depth-First Search (DFS)
3. Uniform-Cost Search (UCS)
4. Depth-Limited Search
5. Iterative Deepening Depth-First Search (IDDFS)
6. Bidirectional Search

### Evaluation Criteria

We evaluate search algorithms on:

1. **Completeness**: Does it always find a solution if one exists?
2. **Optimality**: Does it find the least-cost solution?
3. **Time Complexity**: How many nodes are generated/expanded?
4. **Space Complexity**: Maximum nodes stored in memory?

Complexity is typically expressed using:
- **b**: branching factor (max successors)
- **d**: depth of shallowest goal
- **m**: maximum depth of search tree

### 2.2.1 Generic Search Algorithm

All uninformed search algorithms follow a common template that differs only in how nodes are selected from the frontier:

```
Algorithm: GenericSearch(Initial State s, Goal Condition G)
Input: Initial state s, Goal condition G
Output: Path from s to goal, or failure

begin
    LIST ← {s}                    // Initialize frontier with initial state
    VISIT ← ∅                     // Initialize visited set
    pred ← empty array            // Track predecessors for path reconstruction
    
    repeat
        if LIST is empty then
            return failure         // No solution exists
        
        Select node i from LIST based on strategy  // KEY DIFFERENCE BETWEEN ALGORITHMS
        Delete i from LIST
        Add i to VISIT
        
        if i satisfies G then
            return reconstruct_path(pred, s, i)  // Solution found
        
        for each node j ∈ A(i) do   // A(i) = adjacency list of i
            if j ∉ VISIT then
                Add j to LIST
                pred[j] ← i
        
    until LIST is empty or goal found
end
```

**Key Components:**

- **LIST**: Frontier of nodes to explore (strategy determines structure)
- **VISIT**: Hash table of visited nodes (avoid cycles)
- **pred**: Predecessor array for path reconstruction
- **Strategy**: How nodes are selected from LIST defines the algorithm

### 2.2.2 Breadth-First Search (BFS)

BFS explores the shallowest nodes first, expanding all nodes at depth *d* before any nodes at depth *d+1*.

**Selection Strategy:**
- LIST implemented as **FIFO queue** (First-In-First-Out)
- Select oldest node (earliest added to LIST)

**Algorithm Characteristics:**

```
Strategy: Select earliest node added to LIST (FIFO order)
Data Structure: Queue
Exploration Pattern: Layer-by-layer, breadth-wise
```

**Properties:**

| Property | Value | Explanation |
|----------|-------|-------------|
| Complete | Yes | Finds solution if b is finite |
| Optimal | Yes* | *Only if all action costs equal |
| Time | O(b^d) | Exponential in depth |
| Space | O(b^d) | Must store all nodes at current level |

**Advantages:**
- Finds shallowest solution
- Simple to implement
- Guaranteed to find solution if one exists

**Disadvantages:**
- Memory requirements can be prohibitive
- Explores many irrelevant states
- Slow for deep solutions

**When to Use:**
- Solution is shallow
- State space is small
- Optimality (shortest path) is required
- All actions have same cost

### 2.2.3 Depth-First Search (DFS)

DFS explores the deepest node first, following a single path until reaching a dead end, then backtracking.

**Selection Strategy:**
- LIST implemented as **LIFO stack** (Last-In-First-Out)
- Select newest node (most recently added to LIST)

**Algorithm Characteristics:**

```
Strategy: Select most recent node added to LIST (LIFO order)
Data Structure: Stack
Exploration Pattern: Deep paths first, then backtrack
```

**Properties:**

| Property | Value | Explanation |
|----------|-------|-------------|
| Complete | No | Can get stuck in infinite paths |
| Optimal | No | May find suboptimal solutions |
| Time | O(b^m) | Can be exponential in max depth |
| Space | O(bm) | Only stores path + siblings |

**Advantages:**
- Low memory requirements
- Can find solutions quickly if lucky
- Suitable for problems with many solutions

**Disadvantages:**
- May not terminate in infinite state spaces
- Can find very long paths
- Not optimal

**When to Use:**
- Memory is limited
- State space is deep but finite
- Any solution is acceptable (optimality not required)

### 2.2.4 Uniform-Cost Search (UCS)

UCS is a variant of BFS that handles weighted graphs, always expanding the node with lowest path cost.

**Selection Strategy:**
- LIST implemented as **priority queue** ordered by path cost g(n)
- Select node with minimum path cost from start

**Modified Generic Algorithm:**

```
Enhancement to GenericSearch:

Initialize: c[s] ← 0, c[all others] ← ∞

When expanding node i:
    for each j ∈ A(i) with edge cost d(i,j):
        if j ∉ VISIT:
            if c[i] + d(i,j) < c[j]:      // Found better path
                c[j] ← c[i] + d(i,j)
                pred[j] ← i
                Add/Update j in priority queue with priority c[j]

Selection: Choose node with minimum c[·] from LIST
```

**Properties:**

| Property | Value | Explanation |
|----------|-------|-------------|
| Complete | Yes | If all costs positive |
| Optimal | Yes | Always finds least-cost path |
| Time | O(b^(C*/ε)) | C* = optimal cost, ε = min edge cost |
| Space | O(b^(C*/ε)) | Stores all explored nodes |

**Key Insight:**
UCS is essentially Dijkstra's algorithm applied to search. It explores nodes in order of increasing path cost from the start.

**When to Use:**
- Action costs vary
- Need optimal solution
- State space has weighted edges

### 2.2.5 Iterative Deepening DFS (IDDFS)

IDDFS combines the space efficiency of DFS with the optimality of BFS by performing depth-limited searches with incrementally increasing depth limits.

**Algorithm:**

```
Algorithm: IDDFS(Initial State s, Goal Condition G)

begin
    for depth_limit = 0, 1, 2, ... do
        result ← DLS(s, G, depth_limit)  // Depth-Limited Search
        if result ≠ cutoff then
            return result
end

Algorithm: DLS(node, Goal G, limit)

begin
    if node satisfies G then
        return node
    else if limit = 0 then
        return cutoff
    else
        cutoff_occurred ← false
        for each successor of node do
            result ← DLS(successor, G, limit-1)
            if result = cutoff then
                cutoff_occurred ← true
            else if result ≠ failure then
                return result
        if cutoff_occurred then
            return cutoff
        else
            return failure
end
```

**Properties:**

| Property | Value | Explanation |
|----------|-------|-------------|
| Complete | Yes | For finite branching factor |
| Optimal | Yes | For uniform costs |
| Time | O(b^d) | Similar to BFS |
| Space | O(bd) | Linear space! |

**Why IDDFS is Practical:**

Although IDDFS repeats work, the overhead is modest:
- Most nodes are at the bottom level
- Total nodes generated: b^d + b^(d-1) + ... + b + 1
- Repeated work at higher levels is negligible

**When to Use:**
- Unknown depth
- Limited memory
- Need optimal solution
- Best of both worlds: BFS optimality + DFS space efficiency

### 2.2.6 Bidirectional Search

Bidirectional search simultaneously searches forward from the initial state and backward from the goal state until the two searches meet.

**Algorithm:**

```
Algorithm: BidirectionalSearch(Initial State s, Goal State g)

begin
    FLIST ← {s}              // Forward frontier
    BLIST ← {g}              // Backward frontier
    FVISIT ← ∅              // Forward visited
    BVISIT ← ∅              // Backward visited
    
    repeat
        // Forward expansion
        Select node i_f from FLIST
        Delete i_f from FLIST
        Add i_f to FVISIT
        
        for each j ∈ A(i_f) not in FVISIT do
            Add j to FLIST
            pred[j] ← i_f
        
        // Backward expansion
        Select node i_b from BLIST
        Delete i_b from BLIST
        Add i_b to BVISIT
        
        for each j ∈ B(i_b) not in BVISIT do  // B(i) = reverse adjacency
            Add j to BLIST
            succ[j] ← i_b
        
        // Check for intersection
        if FLIST ∩ BLIST ≠ ∅ then
            common_node ← any node in FLIST ∩ BLIST
            return reconstruct_bidirectional_path(pred, succ, s, common_node, g)
    
    until FLIST is empty or BLIST is empty
    
    return failure
end
```

**Key Advantage:**

If both searches use BFS and meet at depth d/2:
- Unidirectional BFS: O(b^d) nodes
- Bidirectional BFS: 2×O(b^(d/2)) = O(b^(d/2)) nodes

This is a **massive** saving! For b=10, d=6: 10^6 vs. 2×10^3

**Challenges:**
- Requires explicit goal state(s)
- Computing intersection can be expensive (use hash table)
- Backward operators must be derivable
- Both searches must use compatible strategies

## 2.3 Informed Search Algorithms

**Informed** (or **heuristic**) search uses problem-specific knowledge to guide search toward goals more efficiently.

### Heuristic Function

A **heuristic function** h(n) estimates the cost from node n to the nearest goal.

**Properties:**

1. **h(n) ≥ 0** for all nodes n
2. **h(goal) = 0** for all goal nodes
3. **Admissible**: h(n) ≤ h*(n) (never overestimates)
   - h*(n) = true optimal cost from n to goal
4. **Consistent** (Monotonic): h(n) ≤ c(n,n') + h(n')
   - For every edge from n to n' with cost c(n,n')
   - Implies admissibility

### Evaluation Functions

Informed search uses combinations of:
- **g(n)**: Actual cost from start to n
- **h(n)**: Estimated cost from n to goal
- **f(n)**: Estimated total cost through n

Different algorithms use different formulas for f(n).

### 2.3.1 Greedy Best-First Search

Greedy Best-First Search always expands the node that appears closest to the goal according to the heuristic.

**Evaluation Function:**

$$f(n) = h(n)$$

Only uses heuristic estimate, ignores path cost.

**Algorithm:**

```
Selection Strategy: Choose node i with minimum h(i) from LIST
Data Structure: Priority queue ordered by h(n)
```

**Properties:**

| Property | Value | Explanation |
|----------|-------|-------------|
| Complete | No | Can get stuck in loops |
| Optimal | No | Ignores path costs |
| Time | O(b^m) | Worst case, can be much better |
| Space | O(b^m) | Keeps all nodes in memory |

**Characteristics:**

- **Fast**: Often finds solutions quickly
- **Risky**: Can be led astray by misleading heuristic
- **Greedy**: Makes locally optimal choices

**Example - 8-Puzzle:**

Using Manhattan distance heuristic, greedy search always moves tiles closer to their goal positions, even if it increases path length.

**When to Use:**
- Have good heuristic
- Want fast solutions
- Optimality not critical
- Path cost not important

### 2.3.2 A* Search

A* is the most widely used informed search algorithm. It combines actual cost with heuristic estimate.

**Evaluation Function:**

$$f(n) = g(n) + h(n)$$

Where:
- **g(n)**: Actual cost from start to n
- **h(n)**: Estimated cost from n to goal
- **f(n)**: Estimated total cost of cheapest solution through n

**Algorithm:**

```
Algorithm: A*(Initial State s, Goal Condition G, Heuristic h)

begin
    g[s] ← 0
    g[all others] ← ∞
    LIST ← priority queue with s, priority f(s) = g(s) + h(s)
    VISIT ← ∅
    
    while LIST not empty do
        n ← node in LIST with minimum f(n)
        Remove n from LIST
        
        if n satisfies G then
            return reconstruct_path(n)  // Optimal solution!
        
        Add n to VISIT
        
        for each successor n' of n with edge cost c do
            if n' in VISIT then
                continue  // Already processed
            
            tentative_g ← g[n] + c
            
            if tentative_g < g[n'] then  // Found better path
                g[n'] ← tentative_g
                f[n'] ← g[n'] + h(n')
                pred[n'] ← n
                
                if n' not in LIST then
                    Add n' to LIST with priority f[n']
                else
                    Update n' priority to f[n']
    
    return failure
end
```

**Properties (with admissible h):**

| Property | Value | Explanation |
|----------|-------|-------------|
| Complete | Yes | Always finds solution if exists |
| Optimal | Yes | Guaranteed least-cost path! |
| Time | Exponential | But much better than uninformed |
| Space | Exponential | Keeps all nodes (can be limiting) |

### Why A* is Optimal

**Theorem**: If h(n) is admissible, A* is optimal.

**Proof Sketch:**

1. Suppose A* returned suboptimal goal G₂ with cost C₂
2. Let G be optimal goal with cost C* < C₂
3. There must be some unexpanded node n on optimal path to G
4. Since h is admissible: f(n) = g(n) + h(n) ≤ C*
5. But A* expanded G₂, so f(G₂) ≤ f(n)
6. Since h(G₂) = 0: f(G₂) = g(G₂) = C₂
7. Therefore: C₂ ≤ f(n) ≤ C*
8. Contradiction! Cannot have C* < C₂ and C₂ ≤ C*

∴ A* always returns optimal solution when h is admissible.

### Consistency (Monotonicity)

A stronger property than admissibility:

**h(n) ≤ c(n,n') + h(n')** for all edges n→n'

**Benefits:**
- Ensures f(n) never decreases along any path
- Can check goal when generated (not just when expanded)
- Guarantees each state expanded at most once

**Relationship:**
- Consistent ⟹ Admissible
- Admissible ⇏ Consistent (but often is in practice)

### 2.3.3 Heuristics for 8-Puzzle

The 8-puzzle provides excellent examples of heuristic design.

**Problem**: 3×3 grid with 8 numbered tiles and one blank. Slide tiles to reach goal configuration.

**State Space**: 9!/2 = 181,440 reachable states

**Heuristic 1: Misplaced Tiles**

$$h_1(n) = \text{number of tiles not in goal position}$$

- **Admissible?** Yes (each misplaced tile needs ≥1 move)
- **Quality**: Weak, doesn't consider distance

**Heuristic 2: Manhattan Distance**

$$h_2(n) = \sum_{\text{tiles}} (|x_{\text{current}} - x_{\text{goal}}| + |y_{\text{current}} - y_{\text{goal}}|)$$

Sum of horizontal + vertical distances for each tile.

- **Admissible?** Yes (can't move tile to goal faster than Manhattan)
- **Quality**: Better, considers actual distances

**Heuristic Dominance:**

Heuristic h₂ **dominates** h₁ if h₂(n) ≥ h₁(n) for all n, and both are admissible.

For 8-puzzle: h₂(n) ≥ h₁(n) always
- If h₂ dominates h₁, A* with h₂ expands fewer nodes
- Manhattan distance dominates misplaced tiles

**Empirical Results** (8-puzzle from random state to goal):

| Heuristic | Nodes Expanded | Solution Length |
|-----------|----------------|-----------------|
| None (UCS) | ~170,000 | Optimal |
| Misplaced | ~500 | Optimal |
| Manhattan | ~50 | Optimal |

Better heuristics dramatically reduce search!

### 2.3.4 Designing Admissible Heuristics

**Relaxation Method:**

1. Start with original problem
2. Remove constraints to create relaxed problem
3. Optimal cost in relaxed problem = heuristic for original

**Example - 8-Puzzle:**

Original rules: Tile can slide to adjacent empty space

**Relaxation 1**: Tile can move to any adjacent space
- Optimal cost = Manhattan distance

**Relaxation 2**: Tile can move to any space
- Optimal cost = misplaced tiles

**Pattern Databases:**

1. Choose subset of problem (e.g., 4 tiles in 8-puzzle)
2. Pre-compute optimal costs from all configurations
3. Store in database (pattern database)
4. Use as heuristic at runtime

**Combining Heuristics:**

Given admissible h₁, h₂, ..., hₘ:

$$h(n) = \max(h_1(n), h_2(n), ..., h_m(n))$$

The maximum is also admissible and dominates each individual heuristic!

## 2.4 Local Search Algorithms

Local search algorithms operate on **complete state** configurations, making small modifications to improve an objective function.

**Characteristics:**
- Don't maintain search tree or frontier
- Work with single current state
- Move to neighboring states
- Goal: Find state with minimum loss (or maximum utility)

**Applications:**
- Optimization problems
- Constraint satisfaction (N-Queens)
- Machine learning (parameter tuning)
- Scheduling and planning

**Key Difference from Path Search:**
- Path to solution doesn't matter
- Only final state configuration matters
- Memory efficient (constant space)

### 2.4.1 Hill Climbing

Hill climbing is a greedy local search that always moves to the best neighboring state.

**Algorithm:**

```
Algorithm: HillClimbing(Initial State s, Loss Function L)

begin
    current ← s
    
    loop
        neighbors ← generate_neighbors(current)
        
        if neighbors is empty then
            return current  // Local optimum
        
        next ← neighbor with minimum L(neighbor)
        
        if L(next) ≥ L(current) then
            return current  // Can't improve
        
        current ← next
end
```

**Variants:**

1. **Steepest-Ascent**: Examine all neighbors, choose best
2. **First-Choice**: Choose first neighbor that improves
3. **Stochastic**: Choose random improving neighbor
4. **Random-Restart**: Run multiple times with random starts

**Problems:**

1. **Local Maxima**: Better than neighbors but not global optimum
2. **Plateaus**: Flat regions with no gradient
3. **Ridges**: Sequence of local maxima

**Solutions:**

- **Random Restart**: Try multiple initial states
- **Simulated Annealing**: Allow worse moves probabilistically
- **Tabu Search**: Maintain list of recently visited states to avoid

**When to Use:**
- Large state space
- Gradient available
- Local optimum acceptable
- Limited memory

### 2.4.2 Simulated Annealing

Simulated annealing allows occasional moves to worse states to escape local optima, inspired by metallurgical annealing.

**Algorithm:**

```
Algorithm: SimulatedAnnealing(Initial State s, Loss L, Schedule)

begin
    current ← s
    
    for t = 1 to ∞ do
        T ← Schedule(t)  // Temperature decreases over time
        
        if T = 0 then
            return current
        
        next ← random neighbor of current
        ΔL ← L(next) - L(current)
        
        if ΔL < 0 then  // Improvement
            current ← next
        else  // Worse move
            with probability e^(-ΔL/T):
                current ← next  // Accept worse move
end
```

**Key Concepts:**

**Temperature Schedule T(t):**
- High T: Accept many worse moves (exploration)
- Low T: Accept few worse moves (exploitation)
- T → 0: Becomes hill climbing

**Acceptance Probability:**

$$P(\text{accept}) = \begin{cases}
1 & \text{if } \Delta L < 0 \text{ (improvement)} \\
e^{-\Delta L / T} & \text{if } \Delta L \geq 0 \text{ (worse move)}
\end{cases}$$

**Temperature Schedules:**

1. **Linear**: T(t) = T₀ - αt
2. **Exponential**: T(t) = T₀ × γᵗ (γ < 1)
3. **Logarithmic**: T(t) = T₀ / log(t+1)

**Properties:**

- **Completeness**: Finds global optimum with probability → 1 (if schedule slow enough)
- **Convergence**: Guaranteed with logarithmic schedule (too slow in practice)
- **Practical**: Often finds good solutions with exponential schedule

**When to Use:**
- Problem has many local optima
- Want to escape local optima
- Can afford computation time
- No better heuristic available

### 2.4.3 Genetic Algorithms

Genetic algorithms use evolutionary principles: selection, crossover, and mutation to search solution space.

**Algorithm:**

```
Algorithm: GeneticAlgorithm(Population_Size, Fitness, Generations)

begin
    population ← initialize_random_population(Population_Size)
    
    for generation = 1 to Generations do
        // Evaluate fitness
        for each individual in population do
            fitness[individual] ← Fitness(individual)
        
        if termination_condition_met then
            return best individual in population
        
        new_population ← empty
        
        while size(new_population) < Population_Size do
            // Selection
            parent1 ← select_parent(population, fitness)  // Weighted by fitness
            parent2 ← select_parent(population, fitness)
            
            // Crossover
            with probability p_crossover:
                child1, child2 ← crossover(parent1, parent2)
            else:
                child1, child2 ← parent1, parent2
            
            // Mutation
            with probability p_mutation:
                child1 ← mutate(child1)
            with probability p_mutation:
                child2 ← mutate(child2)
            
            add child1, child2 to new_population
        
        population ← new_population
    
    return best individual in population
end
```

**Key Components:**

1. **Representation**: Encode solutions as chromosomes (strings)
2. **Fitness Function**: Evaluate quality of each individual
3. **Selection**: Choose parents based on fitness
   - Roulette wheel: probability ∝ fitness
   - Tournament: best of k random individuals
4. **Crossover**: Combine parent chromosomes
   - Single-point, two-point, uniform
5. **Mutation**: Random changes to maintain diversity
   - Bit flip, swap, inversion

**Parameters:**

- Population size: 50-1000 typically
- Crossover probability: 0.6-0.9
- Mutation probability: 0.01-0.1
- Generations: Problem-dependent

**When to Use:**
- Large discrete search space
- Complex fitness landscape
- Can encode solutions as strings
- Population-based search beneficial

## Summary

This chapter covered fundamental search algorithms for AI problem-solving:

### Algorithm Comparison

**Uninformed Search:**

| Algorithm | Complete | Optimal | Time | Space | Best For |
|-----------|----------|---------|------|-------|----------|
| BFS | Yes | Yes* | O(b^d) | O(b^d) | Shallow solutions |
| DFS | No | No | O(b^m) | O(bm) | Deep search, limited memory |
| UCS | Yes | Yes | O(b^(C*/ε)) | O(b^(C*/ε)) | Weighted graphs |
| IDDFS | Yes | Yes* | O(b^d) | O(bd) | Unknown depth |
| Bidirectional | Yes | Yes* | O(b^(d/2)) | O(b^(d/2)) | Single goal, reversible |

*Optimal for uniform costs

**Informed Search:**

| Algorithm | Complete | Optimal | Best For |
|-----------|----------|---------|----------|
| Greedy | No | No | Fast solutions, good heuristic |
| A* | Yes | Yes* | Optimal paths, admissible h |

*With admissible heuristic

**Local Search:**

| Algorithm | Escapes Local Optima | Memory | Best For |
|-----------|---------------------|--------|----------|
| Hill Climbing | No | O(1) | Convex spaces |
| Simulated Annealing | Yes | O(1) | Many local optima |
| Genetic Algorithms | Yes | O(population) | Discrete optimization |

### Key Takeaways

1. **Problem Formulation Matters**: State representation affects efficiency
2. **Heuristics Are Crucial**: Good heuristics dramatically reduce search
3. **Trade-offs Exist**: Time vs. space, optimality vs. speed
4. **A* is Usually Best**: For path-finding with admissible heuristic
5. **Local Search for Optimization**: When path doesn't matter
6. **No Universal Best**: Algorithm choice depends on problem

### Practical Guidelines

**Use Systematic Search When:**
- Need to find path (not just goal state)
- Optimality important
- State space manageable

**Use A* When:**
- Have good admissible heuristic
- Can afford memory
- Need optimal solution
- **This is the default choice!**

**Use Local Search When:**
- Path to solution doesn't matter
- State space too large for systematic search
- Can define good neighbor function
- Near-optimal solution acceptable

In the next chapter, we'll explore adversarial search for game playing.

## 2.5 Implementation

This section provides complete Python implementations of all search algorithms discussed in this chapter, along with a real-world application to route planning.

The implementations are organized as follows:

1. **Framework Classes**: Generic search infrastructure
2. **Uninformed Search**: BFS, DFS, UCS, IDDFS, Bidirectional
3. **Informed Search**: Greedy Best-First, A*
4. **Local Search**: Hill Climbing, Simulated Annealing, Genetic Algorithms
5. **Example Problems**: 8-Puzzle, Maze, N-Queens
6. **Real-World Application**: City Route Planning with OpenStreetMap

### Usage Instructions

Each implementation includes:
- Complete working code
- Detailed docstrings
- Example usage
- Performance statistics
- Visualization (where applicable)

Navigate to the implementation notebook for hands-on examples: [ch02_search_implementation.ipynb](ch02_search_implementation.ipynb)

## Further Reading

### Textbooks
- Aggarwal, C. C. (2021). *Artificial Intelligence: A Textbook*. Springer. [Chapter 2]
- Russell, S., & Norvig, P. (2020). *Artificial Intelligence: A Modern Approach* (4th ed.). [Chapters 3-4]
- Pearl, J. (1984). *Heuristics: Intelligent Search Strategies*. Addison-Wesley.

### Seminal Papers
- Hart, P. E., Nilsson, N. J., & Raphael, B. (1968). A Formal Basis for the Heuristic Determination of Minimum Cost Paths. *IEEE Transactions on Systems Science and Cybernetics*.
- Korf, R. E. (1985). Depth-first iterative-deepening: An optimal admissible tree search. *Artificial Intelligence*, 27(1), 97-109.
- Kirkpatrick, S., Gelatt, C. D., & Vecchi, M. P. (1983). Optimization by simulated annealing. *Science*, 220(4598), 671-680.

### Online Resources
- [Red Blob Games - Pathfinding](https://www.redblobgames.com/pathfinding/) - Interactive visualizations
- [Stanford CS221](https://stanford.edu/~shervine/teaching/cs-221/)
- [PathFinding.js Visual](https://qiao.github.io/PathFinding.js/visual/)